# Tutorial 1: PriX in 5 Minutes — Secret Sharing & CrypTen-Style Basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mugurbil/prix/blob/main/examples/01_quickstart_mpc_basics.ipynb)

**PriX** is a modern, high-performance JAX/XLA drop-in alternative to [CrypTen](https://github.com/facebookresearch/CrypTen) for **Private Machine Learning and Private LLM Research** via Secure Multi-Party Computation (MPC).

In this notebook, you will learn how to:
1. Install `prix-jax` and launch multi-party computations with `@prix.mpc.run_multiprocess` or `prix.run`.
2. Secret-share data across multiple parties (`src=0`, `src=1`) using additive arithmetic shares ($\mathbb{Z}_{2^{64}}$) and XOR binary shares.
3. Perform private arithmetic, matrix multiplications, reductions, and private comparisons (`>`, `==`, `where`, `relu`).

In [ ]:
!pip install -q prix-jax

## 1. Encrypting Tensors & Inspecting Secret Shares (`prix.array`)

In 2-party MPC (`world_size=2`), a plaintext tensor $x$ is encoded into fixed-point integers ($2^{16}$ scale by default) and split into additive shares $\langle x \rangle_0, \langle x \rangle_1 \in \mathbb{Z}_{2^{64}}$ such that:
$$\langle x \rangle_0 + \langle x \rangle_1 \equiv \lfloor x \cdot 2^{16} \rfloor \pmod{2^{64}}$$
Neither party can learn anything about $x$ from its own share `.share` alone. Calling `.reconstruct()` (or the CrypTen alias `.get_plain_text()`) communicates shares to reconstruct and decode $x$.

In [ ]:
import prix

@prix.mpc.run_multiprocess(world_size=2)
def inspect_shares():
    rank = prix.comm.get().get_rank()
    x = prix.array([1.5, -2.25, 4.0], src=0)
    prix.print(f"[Party {rank}] raw int64 share: {x.share}", in_order=True)
    return x.reconstruct()

reconstructed = inspect_shares()
print("Reconstructed plaintext:", reconstructed)

## 2. Joint Computation Between Two Data Owners (`src=0` and `src=1`)

Suppose **Party 0** (e.g., a hospital) holds a private feature vector `x` and **Party 1** (e.g., a model provider) holds private weights `W` and bias `b`. Using `src=0` and `src=1`, each party secret-shares its own tensor without revealing it to the other party:

In [ ]:
@prix.mpc.run_multiprocess(world_size=2)
def two_party_linear_layer():
    rank = prix.comm.get().get_rank()

    # Only Party 0 knows the true values of x; Party 1 passes a dummy tensor of the same shape
    my_x = [[1.0, 2.0, -1.0]] if rank == 0 else [[0.0, 0.0, 0.0]]
    x_enc = prix.array(my_x, src=0)

    # Only Party 1 knows the true weights W and bias b
    my_W = [[0.5, -1.0], [1.5, 0.25], [-0.5, 2.0]] if rank == 1 else [[0.0, 0.0]] * 3
    my_b = [0.1, -0.2] if rank == 1 else [0.0, 0.0]
    W_enc = prix.array(my_W, src=1)
    b_enc = prix.array(my_b, src=1)

    # Compute y = ReLU(x @ W + b) under MPC
    y_enc = (x_enc @ W_enc + b_enc).relu()
    return y_enc.reconstruct()

print("Private ReLU(x @ W + b):", two_party_linear_layer())

## 3. Private Comparisons, `where`, Reductions, and Tensor Factories

PriX supports the full CrypTen/PyTorch-style tensor API (`zeros`, `ones`, `full`, `arange`, `eye`, `rand`, `randn`, `cat`, `stack`, `where`, `>`, `<`, `==`, `sum`, `mean`, `var`, `max`, `argmax`):

In [ ]:
def comparison_and_reduction_demo():
    a = prix.array([-2.0, 0.5, 3.0, -0.25])
    b = prix.array([-1.0, 0.5, 1.0, 2.0])

    # Encrypted comparison returns secret-shared 0.0 / 1.0 mask
    mask = a > b
    # Private multiplexer: select from `a` where a > b, else from `b` (i.e. elementwise max)
    selected = prix.where(mask, a, b)

    return {
        "a > b": mask.reconstruct(),
        "where(a > b, a, b)": selected.reconstruct(),
        "mean(a)": a.mean().reconstruct(),
        "var(a)": a.var(unbiased=False).reconstruct(),
    }

results = prix.run(comparison_and_reduction_demo, world_size=2)
for k, v in results.items():
    print(f"{k:>20}: {v}")